In [ ]:

# ============================================================
# HIPE-2026 MULTITASK OPTIMIZED (AT / isAt) + NER + Optuna
# + Temporal Positional Encoding (T-PE adaptado)
# ------------------------------------------------------------
# ¿Qué se integró del artículo?
# 1) Componente geométrico: positional encoding sinusoidal
#    aplicado sobre la secuencia contextualizada.
# 2) Componente semántico: sesgo de atención basado en
#    similitud entre tokens:
#       S(i,j) = exp(-||x_i - x_j||^2 / (2*sigma^2))
# 3) Refinamiento temporal encima de XLM-RoBERTa mediante
#    una capa transformer temporal ligera.
#
# NOTA IMPORTANTE:
# El artículo original evalúa T-PE en transformers para series
# de tiempo. Aquí se adapta la idea al problema HIPE-2026 de
# extracción de relaciones temporales sobre texto histórico,
# sin reescribir internamente todo XLM-RoBERTa.
# ============================================================

import os
import re
import gc
import time
import json
import math
import random
import warnings
from pathlib import Path
from typing import List, Tuple, Dict, Any, Optional

import numpy as np
import pandas as pd
from pandas.api.types import is_datetime64_any_dtype

from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import transformers
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from tqdm import tqdm

# Optuna (Bayesian)
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

# Plot
try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except Exception:
    HAS_MPL = False

warnings.filterwarnings("ignore")
tqdm.pandas()

# -------------------------
# GPU
# -------------------------
print("torch:", torch.__version__)
print("cuda disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("transformers:", transformers.__version__, transformers.__file__)
from transformers import PreTrainedModel
print("OK: PreTrainedModel")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando:", device)

# -------------------------
# CONFIG BASE (Optuna podrá ajustar varios)
# -------------------------
SEED = 42
MODEL_NAME = "xlm-roberta-base"

# Valores por defecto (si Optuna está ON, luego se sobrescriben)
MAX_LEN = 320
BATCH_SIZE = 8
EPOCHS = 10
LR_ENCODER = 1.5e-5
LR_HEADS = 5e-5
WEIGHT_DECAY = 0.01
DROPOUT = 0.20

# Balanceo Opción 1 (pesos moderados)
AT_CLASS_WEIGHTS_SOFT = {"FALSE": 1.0, "PROBABLE": 2.3, "TRUE": 1.3}

# isAt binario (Optuna lo optimiza)
ISAT_POS_WEIGHT = 3.0

PATIENCE = 2
MAX_GRAD_NORM = 1.0
PIN_MEMORY = torch.cuda.is_available()
NUM_WORKERS = 0
AMP_ENABLED = torch.cuda.is_available()

AT_THR_PROB_INIT = 0.34
AT_THR_TRUE_INIT = 0.50
ISAT_THR_INIT = 0.50

# Balanceo suave (Optuna lo optimiza)
SAMPLER_ALPHA = 0.5

# NER (spaCy)
USE_NER = True

# Clipping para evitar inputs enormes (reduce truncamiento real)
DATELINE_MAX_CHARS = 220
HEADER_MAX_CHARS = 280
EVIDENCE_MAX_CHARS = 900

# ---------- NUEVO: Temporal T-PE ----------
USE_TEMPORAL_TPE = True
TEMPORAL_REFINER_LAYERS = 1
TEMPORAL_REFINER_HEADS = 8
TEMPORAL_SIM_DIM = 64
TEMPORAL_MAX_POS = 512
TEMPORAL_SEMANTIC_WEIGHT = 0.35
TEMPORAL_SIGMA_INIT = 1.0
TEMPORAL_CONTEXT_SCALE = 0.10

# Optuna config
USE_OPTUNA = True
N_TRIALS = 20
EPOCHS_OPTUNA = 3
OPTUNA_TIMEOUT = None

# Paths
DATA_PATHS = [
    r"C:\Users\danil\OneDrive - Universidad Tecnológica de Bolívar\Documentos\clef-2026\HIPE-2026-v1.0-impresso-train-de (1)_UNA_HOJA.xlsx",
    r"C:\Users\danil\OneDrive - Universidad Tecnológica de Bolívar\Documentos\clef-2026\HIPE-2026-v1.0-impresso-train-en_UNA_HOJA.xlsx",
    r"C:\Users\danil\OneDrive - Universidad Tecnológica de Bolívar\Documentos\clef-2026\HIPE-2026-v1.0-impresso-train-fr_UNA_HOJA.xlsx",
]
OUTPUT_DIR = Path("outputs_hipe_multitask_temporal_tpe")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------
# Seed
# -------------------------
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)

# ============================================================
# 1) UTILIDADES
# ============================================================
AT_VALID_LABELS = {"TRUE", "FALSE", "PROBABLE"}
ISAT_VALID_LABELS = {"TRUE", "FALSE"}

AT_LABELS = ["FALSE", "PROBABLE", "TRUE"]
AT_LABEL2ID = {lab: i for i, lab in enumerate(AT_LABELS)}
AT_ID2LABEL = {i: lab for lab, i in AT_LABEL2ID.items()}

ISAT_LABELS = ["FALSE", "TRUE"]
ISAT_LABEL2ID = {"FALSE": 0, "TRUE": 1}
ISAT_ID2LABEL = {0: "FALSE", 1: "TRUE"}

def normalize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df

def normalize_label(x: Any) -> Any:
    if pd.isna(x):
        return np.nan
    s = str(x).strip().upper()
    if s == "PROB":
        s = "PROBABLE"
    return s

def safe_str(x: Any) -> str:
    if pd.isna(x):
        return ""
    return str(x)

def normalize_date_series(s: pd.Series) -> pd.Series:
    s = s.copy()
    if is_datetime64_any_dtype(s):
        out = s.dt.strftime("%Y-%m-%d")
        return out.fillna("UNK").astype(str)

    s_str = s.astype("string").str.strip()
    parsed = pd.to_datetime(s_str, errors="coerce", dayfirst=True)
    out = s_str.copy()
    mask = parsed.notna()
    if mask.any():
        out.loc[mask] = parsed.loc[mask].dt.strftime("%Y-%m-%d")
    return out.fillna("UNK").astype(str).str.strip()

def normalize_ocr_text(text: str) -> str:
    if pd.isna(text):
        return ""
    text = str(text)
    text = text.replace("\r", " ").replace("\n", " ")
    text = re.sub(r"(\w)-\s+(\w)", r"\1\2", text)
    text = (text.replace("“", '"').replace("”", '"')
                .replace("„", '"').replace("’", "'")
                .replace("`", "'"))
    text = re.sub(r"\s+", " ", text).strip()
    return text

def clip_text(s: str, max_chars: int) -> str:
    s = s or ""
    s = re.sub(r"\s+", " ", s).strip()
    if len(s) <= max_chars:
        return s
    return s[:max_chars].rstrip() + "..."

def split_aliases(s: Any) -> List[str]:
    if pd.isna(s):
        return []
    raw = str(s).replace("\n", " ")
    parts = [p.strip() for p in raw.split("|")]
    parts = [p for p in parts if p and p.lower() != "nan"]
    seen = set()
    out = []
    for p in parts:
        p2 = re.sub(r"\s+", " ", p).strip()
        key = p2.lower()
        if key and key not in seen:
            out.append(p2)
            seen.add(key)
    return out

def first_mention(s: Any) -> str:
    aliases = split_aliases(s)
    return aliases[0] if aliases else ""

def simple_sentence_split(text: str) -> List[str]:
    if not text:
        return []
    chunks = re.split(r"(?<=[\.\!\?\:\;])\s+", text)
    chunks = [re.sub(r"\s+", " ", c).strip() for c in chunks if c and c.strip()]
    return chunks

def find_alias_in_sentence(sent: str, aliases: List[str]) -> bool:
    sent_low = sent.lower()
    for a in aliases:
        aa = a.lower().strip()
        if aa and aa in sent_low:
            return True
    return False

def count_alias_hits(sent: str, aliases: List[str]) -> int:
    sent_low = sent.lower()
    c = 0
    for a in aliases:
        aa = a.lower().strip()
        if aa and aa in sent_low:
            c += 1
    return c

# ============================================================
# 2) TEMPORAL CUES
# ============================================================
TEMPORAL_PATTERNS = {
    "en": {
        "past": [r"\bformer\b", r"\bformerly\b", r"\bwas\b", r"\bwere\b", r"\bhad\b",
                 r"\bpreviously\b", r"\bonce\b", r"\bearlier\b", r"\bex-\w+"],
        "present": [r"\bis\b", r"\bare\b", r"\bcurrently\b", r"\bnow\b", r"\bhere\b",
                    r"\btelegraphs\b", r"\bwrites\b", r"\bsays\b", r"\breports\b"],
        "motion": [r"\barrived\b", r"\bleft\b", r"\breturned\b", r"\bcame\b",
                   r"\bwent\b", r"\bvisited\b", r"\bstayed\b", r"\bresided\b"],
        "reporting": [r"\btelegraphs\b", r"\bwrites\b", r"\breports\b", r"\bdispatch\b",
                      r"\baddresses\b", r"\bfrom\s+[A-Z]"],
        "title_words": [r"\barchbishop\b", r"\bbishop\b", r"\bminister\b", r"\bgeneral\b",
                        r"\bpresident\b", r"\bcommander\b", r"\bambassador\b", r"\bdelegat"],
    },
    "fr": {
        "past": [r"\bancien\b", r"\bancienne\b", r"\bétait\b", r"\betaient\b",
                 r"\bavait\b", r"\bautrefois\b", r"\bjadis\b", r"\bex-\w+"],
        "present": [r"\best\b", r"\bsont\b", r"\bmaintenant\b", r"\bactuellement\b",
                    r"\btélégraphie\b", r"\bécrit\b", r"\bannonce\b", r"\bvient de\b"],
        "motion": [r"\best arrivé\b", r"\best parti\b", r"\best revenu\b",
                   r"\ba visité\b", r"\ba séjourné\b", r"\bréside\b"],
        "reporting": [r"\btélégraphie\b", r"\bécrit\b", r"\badresse\b",
                      r"\bdépêche\b", r"\bde\s+[A-ZÀ-ÖØ-Ý]"],
        "title_words": [r"\barchevêque\b", r"\bévêque\b", r"\bministre\b", r"\bgénéral\b",
                        r"\bprésident\b", r"\bchef\b", r"\bdélégu[ée]\b", r"\bambassadeur\b"],
    },
    "de": {
        "past": [r"\behemalig\w*\b", r"\bfrüher\b", r"\bwar\b", r"\bwaren\b",
                 r"\bhatte\b", r"\bgewesen\b"],
        "present": [r"\bist\b", r"\bsind\b", r"\bjetzt\b", r"\bderzeit\b",
                    r"\bmeldet\b", r"\bschreibt\b", r"\bberichtet\b"],
        "motion": [r"\bkam\b", r"\bging\b", r"\breiste\b", r"\bbesuchte\b",
                   r"\bwohnte\b", r"\bkehrte\b", r"\bzurück\b"],
        "reporting": [r"\bmeldet\b", r"\bschreibt\b", r"\bberichtet\b",
                      r"\btelegramm\w*\b", r"\baus\s+[A-ZÄÖÜ]"],
        "title_words": [r"\baußenminister\b", r"\bminister\b", r"\bgeneral\b", r"\bpräsident\b",
                        r"\bbischof\b", r"\berzbischof\b", r"\bbotschaft\b", r"\bdelegiert\w*\b"],
    }
}

PREP_PATTERNS = {
    "generic_from": [r"\bfrom\b", r"\bde\b", r"\bd['’]\b", r"\baus\b", r"\bvon\b", r"\bdepuis\b"],
    "generic_in":   [r"\bin\b", r"\ben\b", r"\bim\b", r"\bà\b", r"\bau\b", r"\bbei\b", r"\bnach\b"],
    "generic_of":   [r"\bof\b", r"\bde\b", r"\bdu\b", r"\bdes\b", r"\bvon\b"],
    "generic_near": [r"\bnear\b", r"\bprès de\b", r"\bnahe\b", r"\bbei\b", r"\bautour de\b"],
}

def regex_any(text: str, patterns: List[str]) -> int:
    for p in patterns:
        if re.search(p, text, flags=re.IGNORECASE):
            return 1
    return 0

def get_lang_family(lang: str) -> str:
    lang = (lang or "").lower().strip()
    if lang.startswith("fr"):
        return "fr"
    if lang.startswith("de"):
        return "de"
    return "en"

def extract_header_and_dateline(text_norm: str) -> Tuple[str, str]:
    sents = simple_sentence_split(text_norm)
    if not sents:
        return "", ""
    header = " ".join(sents[:2])
    dateline = sents[0]
    return clip_text(header, HEADER_MAX_CHARS), clip_text(dateline, DATELINE_MAX_CHARS)

def compute_temporal_and_semantic_cues(
    evidence_text: str,
    dateline_text: str,
    header_text: str,
    pers_aliases: List[str],
    loc_aliases: List[str],
    lang: str
) -> Dict[str, int]:
    lang_key = get_lang_family(lang)
    patt = TEMPORAL_PATTERNS[lang_key]

    ev_low = evidence_text.lower()
    dt_low = dateline_text.lower()

    cue_from = regex_any(ev_low, PREP_PATTERNS["generic_from"])
    cue_in   = regex_any(ev_low, PREP_PATTERNS["generic_in"])
    cue_of   = regex_any(ev_low, PREP_PATTERNS["generic_of"])
    cue_near = regex_any(ev_low, PREP_PATTERNS["generic_near"])

    has_past_cue = regex_any(ev_low, patt["past"])
    has_present_cue = regex_any(ev_low, patt["present"])
    has_motion_verb = regex_any(ev_low, patt["motion"])
    has_reporting_verb = regex_any(ev_low, patt["reporting"])

    has_title_word = regex_any(ev_low, patt["title_words"])
    has_title_apposition = int(has_title_word and (cue_of or cue_in))

    loc_in_dateline = int(any((a.lower() in dt_low) for a in loc_aliases if a))
    dateline_from_cue = regex_any(dt_low, PREP_PATTERNS["generic_from"])
    has_dateline_from_place = int(loc_in_dateline and dateline_from_cue)

    if has_dateline_from_place or (has_present_cue and has_reporting_verb):
        temp_bucket = 2
    elif has_past_cue or (has_motion_verb and not has_present_cue):
        temp_bucket = 1
    else:
        temp_bucket = 0

    return {
        "cue_from": cue_from, "cue_in": cue_in, "cue_of": cue_of, "cue_near": cue_near,
        "has_past_cue": has_past_cue, "has_present_cue": has_present_cue,
        "has_motion_verb": has_motion_verb, "has_reporting_verb": has_reporting_verb,
        "has_title_apposition": has_title_apposition, "has_dateline_from_place": has_dateline_from_place,
        "temp_bucket": temp_bucket,
    }

def extract_evidence_window(
    text: str,
    pers_aliases: List[str],
    loc_aliases: List[str],
    lang: str,
    window: int = 1
) -> Tuple[str, str, str, Dict[str, int]]:
    text_norm = normalize_ocr_text(text)
    sents = simple_sentence_split(text_norm)
    header_text, dateline_text = extract_header_and_dateline(text_norm)

    if not sents:
        feats = {"same_sentence": 0, "distance_bucket": 3, "pers_hit_count_ev": 0, "loc_hit_count_ev": 0}
        feats.update(compute_temporal_and_semantic_cues("", dateline_text, header_text, pers_aliases, loc_aliases, lang))
        return "", dateline_text, header_text, feats

    pers_idxs, loc_idxs = [], []
    for i, s in enumerate(sents):
        if find_alias_in_sentence(s, pers_aliases):
            pers_idxs.append(i)
        if find_alias_in_sentence(s, loc_aliases):
            loc_idxs.append(i)

    same_idx = None
    for i in pers_idxs:
        if i in loc_idxs:
            same_idx = i
            break

    if same_idx is not None:
        start = max(0, same_idx - window)
        end = min(len(sents), same_idx + window + 1)
        evidence_text = " ".join(sents[start:end])
        min_dist = 0
        same_sentence = 1
    else:
        if pers_idxs and loc_idxs:
            best_pair = None
            best_dist = 10**9
            for i in pers_idxs:
                for j in loc_idxs:
                    d = abs(i - j)
                    if d < best_dist:
                        best_dist = d
                        best_pair = (i, j)
            i, j = best_pair
            start = max(0, min(i, j) - window)
            end = min(len(sents), max(i, j) + window + 1)
            evidence_text = " ".join(sents[start:end])
            min_dist = best_dist
            same_sentence = 0
        else:
            evidence_text = " ".join(sents[:min(3, len(sents))])
            min_dist = 99
            same_sentence = 0

    if min_dist == 0:
        dist_bucket = 0
    elif min_dist == 1:
        dist_bucket = 1
    elif min_dist <= 3:
        dist_bucket = 2
    else:
        dist_bucket = 3

    evidence_text = clip_text(evidence_text, EVIDENCE_MAX_CHARS)

    pers_hit_count_ev = count_alias_hits(evidence_text, pers_aliases)
    loc_hit_count_ev = count_alias_hits(evidence_text, loc_aliases)

    feats = {
        "same_sentence": same_sentence,
        "distance_bucket": dist_bucket,
        "pers_hit_count_ev": pers_hit_count_ev,
        "loc_hit_count_ev": loc_hit_count_ev,
    }
    feats.update(compute_temporal_and_semantic_cues(evidence_text, dateline_text, header_text, pers_aliases, loc_aliases, lang))
    return evidence_text, dateline_text, header_text, feats

def temp_bucket_to_token(x: int) -> str:
    return {0: "UNCERTAIN", 1: "PAST", 2: "IMMEDIATE"}.get(int(x), "UNCERTAIN")

# ============================================================
# 3) NER (spaCy)
# ============================================================
NER_AVAILABLE = False
try:
    import spacy
    NER_AVAILABLE = True
except Exception:
    NER_AVAILABLE = False

_NLP_CACHE = {}

def load_spacy_model(lang: str):
    if not NER_AVAILABLE:
        return None
    lang_key = get_lang_family(lang)
    if lang_key in _NLP_CACHE:
        return _NLP_CACHE[lang_key]

    candidates = {
        "en": ["en_core_web_sm", "xx_ent_wiki_sm"],
        "fr": ["fr_core_news_sm", "xx_ent_wiki_sm"],
        "de": ["de_core_news_sm", "xx_ent_wiki_sm"],
    }[lang_key]

    nlp = None
    for name in candidates:
        try:
            nlp = spacy.load(name, disable=["tagger", "parser", "lemmatizer", "attribute_ruler"])
            break
        except Exception:
            nlp = None

    _NLP_CACHE[lang_key] = nlp
    if nlp is None:
        print(f"⚠️ spaCy NER no disponible para lang={lang_key}. Instala modelos spaCy.")
    return nlp

def ner_features(text: str, lang: str, target_pers: str, target_loc: str) -> Dict[str, int]:
    if not USE_NER:
        return {"NER_PER": 0, "NER_LOC": 0, "NER_TPER": 0, "NER_TLOC": 0, "NER_OP": 0, "NER_OL": 0}

    nlp = load_spacy_model(lang)
    if nlp is None:
        t = (text or "").lower()
        tp = int(bool(target_pers and target_pers.lower() in t))
        tl = int(bool(target_loc and target_loc.lower() in t))
        return {"NER_PER": 0, "NER_LOC": 0, "NER_TPER": tp, "NER_TLOC": tl, "NER_OP": 0, "NER_OL": 0}

    doc = nlp(text[:2000])
    per_labels = {"PERSON", "PER"}
    loc_labels = {"GPE", "LOC"}

    persons = []
    places = []
    for ent in doc.ents:
        if ent.label_ in per_labels:
            persons.append(ent.text)
        elif ent.label_ in loc_labels:
            places.append(ent.text)

    persons_low = [p.lower().strip() for p in persons]
    places_low = [p.lower().strip() for p in places]
    tp = target_pers.lower().strip() if target_pers else ""
    tl = target_loc.lower().strip() if target_loc else ""

    target_person_found = int(tp != "" and any(tp in p for p in persons_low))
    target_place_found = int(tl != "" and any(tl in p for p in places_low))

    n_other_persons = int(max(0, len(set(persons_low)) - target_person_found))
    n_other_places = int(max(0, len(set(places_low)) - target_place_found))

    return {
        "NER_PER": int(len(set(persons_low))),
        "NER_LOC": int(len(set(places_low))),
        "NER_TPER": target_person_found,
        "NER_TLOC": target_place_found,
        "NER_OP": n_other_persons,
        "NER_OL": n_other_places,
    }

# ============================================================
# 4) INPUT PAIR-CENTRIC + TEMPORAL FEATURE VECTOR
# ============================================================
TEMPORAL_FEATURE_NAMES = [
    "cue_from", "cue_in", "cue_of", "cue_near",
    "same_sentence", "distance_bucket_norm",
    "pers_hits_norm", "loc_hits_norm",
    "has_dateline_from_place",
    "has_past_cue", "has_present_cue", "has_motion_verb", "has_reporting_verb",
    "has_title_apposition", "temp_bucket_norm",
    "ner_per_norm", "ner_loc_norm",
    "ner_tper", "ner_tloc",
    "ner_op_norm", "ner_ol_norm",
    "year_norm", "month_sin", "month_cos",
]

def parse_year_month(date_str: str) -> Tuple[float, float, float]:
    date_str = safe_str(date_str).strip()
    try:
        dt = pd.to_datetime(date_str, errors="coerce")
    except Exception:
        dt = pd.NaT

    if pd.isna(dt):
        return 0.0, 0.0, 0.0

    year = float(dt.year)
    month = float(dt.month)

    year_norm = (year - 1800.0) / 250.0
    year_norm = max(-1.0, min(1.5, year_norm))

    angle = 2.0 * math.pi * (month / 12.0)
    month_sin = math.sin(angle)
    month_cos = math.cos(angle)
    return year_norm, month_sin, month_cos

def build_paircentric_package(row: pd.Series) -> Dict[str, Any]:
    lang = safe_str(row.get("language", "UNK"))
    date = safe_str(row.get("date", "UNK"))
    pers = first_mention(row.get("pers_mentions_list", ""))
    loc = first_mention(row.get("loc_mentions_list", ""))

    text_raw = row.get("text", "")
    pers_aliases = split_aliases(row.get("pers_mentions_list", ""))
    loc_aliases = split_aliases(row.get("loc_mentions_list", ""))

    ev_text, dateline_text, header_text, feats = extract_evidence_window(
        text=text_raw,
        pers_aliases=pers_aliases,
        loc_aliases=loc_aliases,
        lang=lang,
        window=1
    )

    ner = ner_features(ev_text, lang=lang, target_pers=pers, target_loc=loc)
    temp_tok = temp_bucket_to_token(feats["temp_bucket"])

    cues_block = (
        f"[CUES "
        f"FROM={feats['cue_from']} IN={feats['cue_in']} OF={feats['cue_of']} NEAR={feats['cue_near']} "
        f"SAME_SENT={feats['same_sentence']} DIST={feats['distance_bucket']} "
        f"PERS_HITS={feats['pers_hit_count_ev']} LOC_HITS={feats['loc_hit_count_ev']} "
        f"DATELINE_FROM_PLACE={feats['has_dateline_from_place']} "
        f"PAST={feats['has_past_cue']} PRESENT={feats['has_present_cue']} "
        f"MOTION={feats['has_motion_verb']} REPORT={feats['has_reporting_verb']} "
        f"APPOS={feats['has_title_apposition']} TEMP={temp_tok} "
        f"NER_PER={ner['NER_PER']} NER_LOC={ner['NER_LOC']} "
        f"NER_TPER={ner['NER_TPER']} NER_TLOC={ner['NER_TLOC']} "
        f"NER_OP={ner['NER_OP']} NER_OL={ner['NER_OL']}]"
    )

    model_input = (
        f"[LANG={lang}] [DATE={date}] [TASK=AT+ISAT] "
        f"[PERS] {pers} [/PERS] [LOC] {loc} [/LOC] "
        f"{cues_block} "
        f"[DATELINE] {dateline_text} [/DATELINE] "
        f"[HEADER] {header_text} [/HEADER] "
        f"[EVIDENCE] {ev_text} [/EVIDENCE]"
    )
    model_input = re.sub(r"\s+", " ", model_input).strip()

    year_norm, month_sin, month_cos = parse_year_month(date)

    temporal_vector = np.array([
        float(feats["cue_from"]),
        float(feats["cue_in"]),
        float(feats["cue_of"]),
        float(feats["cue_near"]),
        float(feats["same_sentence"]),
        float(feats["distance_bucket"]) / 3.0,
        min(float(feats["pers_hit_count_ev"]) / 4.0, 1.0),
        min(float(feats["loc_hit_count_ev"]) / 4.0, 1.0),
        float(feats["has_dateline_from_place"]),
        float(feats["has_past_cue"]),
        float(feats["has_present_cue"]),
        float(feats["has_motion_verb"]),
        float(feats["has_reporting_verb"]),
        float(feats["has_title_apposition"]),
        float(feats["temp_bucket"]) / 2.0,
        min(float(ner["NER_PER"]) / 8.0, 1.0),
        min(float(ner["NER_LOC"]) / 8.0, 1.0),
        float(ner["NER_TPER"]),
        float(ner["NER_TLOC"]),
        min(float(ner["NER_OP"]) / 8.0, 1.0),
        min(float(ner["NER_OL"]) / 8.0, 1.0),
        float(year_norm),
        float(month_sin),
        float(month_cos),
    ], dtype=np.float32)

    return {
        "model_input": model_input,
        "temporal_feats": temporal_vector,
        "evidence_text": ev_text,
        "dateline_text": dateline_text,
        "header_text": header_text,
    }

# ============================================================
# 5) split + distribuciones
# ============================================================
def print_distributions(title: str, data: pd.DataFrame):
    print(f"\n{'='*26} {title} {'='*26}")
    print("AT (count):")
    print(data["at"].value_counts(dropna=False))
    print("\nisAt (count):")
    print(data["isAt"].value_counts(dropna=False))
    print("\nCombo (at | isAt):")
    print(data["combo_label"].value_counts(dropna=False))
    print("\nDocs / Filas:", data["document_id"].nunique(), "/", len(data))
    print("=" * 80)

def safe_group_stratified_split(df: pd.DataFrame, test_size=0.30, random_state=42):
    doc_df = (
        df.groupby("document_id")
          .agg(
              doc_strat_combo=("combo_label", lambda x: x.mode().iloc[0] if len(x.mode()) else x.iloc[0]),
              doc_strat_at=("at", lambda x: x.mode().iloc[0] if len(x.mode()) else x.iloc[0]),
              doc_strat_isat=("isAt", lambda x: x.mode().iloc[0] if len(x.mode()) else x.iloc[0]),
              n_pairs=("document_id", "size"),
          )
          .reset_index()
    )

    stratify_cols = ["doc_strat_combo", "doc_strat_at", None]
    for strat_col in stratify_cols:
        try:
            stratify_vals = doc_df[strat_col] if strat_col is not None else None
            train_docs, val_docs = train_test_split(
                doc_df["document_id"],
                test_size=test_size,
                random_state=random_state,
                stratify=stratify_vals
            )
            print(f"✅ Split realizado con estratificación por: {strat_col if strat_col else 'None (fallback)'}")
            return train_docs, val_docs, doc_df
        except Exception as e:
            print(f"⚠️ No se pudo estratificar por {strat_col}: {e}")
    raise RuntimeError("No fue posible realizar el split.")

def soft_balance_sampler_from_combo(train_df: pd.DataFrame, alpha: float = 0.5) -> WeightedRandomSampler:
    combo_counts = train_df["combo_label"].value_counts().to_dict()
    N = len(train_df)
    weights = np.array([(N / combo_counts[c]) ** alpha for c in train_df["combo_label"].tolist()], dtype=np.float32)
    sampler = WeightedRandomSampler(
        weights=torch.tensor(weights, dtype=torch.double),
        num_samples=len(train_df),
        replacement=True
    )
    return sampler

def simulate_sampler_distribution(train_df: pd.DataFrame, alpha: float, draws: int = None, seed: int = 42) -> pd.DataFrame:
    if draws is None:
        draws = len(train_df)
    combo_counts = train_df["combo_label"].value_counts().to_dict()
    N = len(train_df)
    weights = np.array([(N / combo_counts[c]) ** alpha for c in train_df["combo_label"].tolist()], dtype=np.float64)

    g = np.random.default_rng(seed)
    idxs = g.choice(np.arange(len(train_df)), size=draws, replace=True, p=weights / weights.sum())
    sim = train_df.iloc[idxs]
    return sim["combo_label"].value_counts().rename_axis("combo_label").reset_index(name="count")

# ============================================================
# 6) Threshold tuning
# ============================================================
def probs_to_at_preds_with_thresholds(probs_at: np.ndarray, thr_prob: float = 0.34, thr_true: float = 0.50) -> np.ndarray:
    preds = []
    for p in probs_at:
        p_false, p_prob, p_true = float(p[0]), float(p[1]), float(p[2])
        base = int(np.argmax(p))
        if base == 1 and p_prob < thr_prob:
            pred = 2 if p_true >= p_false else 0
        elif base == 2 and p_true < thr_true:
            pred = 1 if p_prob >= p_false else 0
        else:
            pred = base
        preds.append(pred)
    return np.array(preds, dtype=int)

def tune_at_thresholds_for_ba(y_true_at: np.ndarray, probs_at: np.ndarray) -> Dict[str, Any]:
    grid_prob = [0.20, 0.25, 0.30, 0.34, 0.38, 0.42, 0.46, 0.50, 0.55]
    grid_true = [0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]
    best = {"thr_prob": 0.34, "thr_true": 0.50, "ba_at": -1.0}
    for tp in grid_prob:
        for tt in grid_true:
            pred = probs_to_at_preds_with_thresholds(probs_at, thr_prob=tp, thr_true=tt)
            ba = balanced_accuracy_score(y_true_at, pred)
            if ba > best["ba_at"]:
                best = {"thr_prob": tp, "thr_true": tt, "ba_at": float(ba)}
    return best

def tune_isat_threshold_for_ba(y_true_isat: np.ndarray, probs_isat: np.ndarray) -> Dict[str, Any]:
    grid = [round(x, 2) for x in np.arange(0.15, 0.86, 0.02)]
    best = {"thr": 0.50, "ba_isat": -1.0}
    for t in grid:
        pred = (probs_isat >= t).astype(int)
        ba = balanced_accuracy_score(y_true_isat, pred)
        if ba > best["ba_isat"]:
            best = {"thr": float(t), "ba_isat": float(ba)}
    return best

def compute_efficiency_proxy(acc_macro: float, model_size_mb: float, time_per_sample_ms: float, using_cuda: bool, opensource_flag: int = 1) -> float:
    speed_score = 1.0 / (1.0 + time_per_sample_ms)
    size_score = 1.0 / (1.0 + model_size_mb / 100.0)
    hw_score = 0.8 if using_cuda else 0.5
    eff_proxy = (
        0.60 * acc_macro +
        0.15 * speed_score +
        0.10 * size_score +
        0.10 * hw_score +
        0.05 * opensource_flag
    )
    return float(eff_proxy)

# ============================================================
# 7) CARGA DATA
# ============================================================
dfs = []
for p in DATA_PATHS:
    p_obj = Path(p)
    if not p_obj.exists():
        raise FileNotFoundError(f"No existe el archivo: {p}")
    tmp = pd.read_excel(p_obj)
    tmp = normalize_column_names(tmp)
    tmp["source_file"] = p_obj.name
    dfs.append(tmp)
    print(f"✅ Cargado: {p_obj.name} | shape={tmp.shape}")

df = pd.concat(dfs, ignore_index=True)
df = normalize_column_names(df)

required_cols = ["document_id", "date", "language", "pers_mentions_list", "loc_mentions_list", "at", "isAt", "text"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Faltan columnas requeridas: {missing}")

df["at"] = df["at"].apply(normalize_label)
df["isAt"] = df["isAt"].apply(normalize_label)

before_rows = len(df)
df = df[df["at"].isin(AT_VALID_LABELS) & df["isAt"].isin(AT_VALID_LABELS | ISAT_VALID_LABELS)].copy()
after_rows = len(df)
if after_rows < before_rows:
    print(f"⚠️ Se eliminaron {before_rows - after_rows} filas por labels inválidos/nulos.")

unique_isat = set(df["isAt"].dropna().unique().tolist())
if "PROBABLE" in unique_isat:
    raise ValueError("Este script asume isAt binario; tu dataset tiene PROBABLE en isAt.")

df["text"] = df["text"].fillna("").astype(str)
df["language"] = df["language"].astype("string").str.strip().str.lower().fillna("unk")
df["date"] = normalize_date_series(df["date"])
df["document_id"] = df["document_id"].astype("string").fillna("UNK_DOC").astype(str)
df["combo_label"] = df["at"].astype(str) + " | " + df["isAt"].astype(str)

df.to_csv(OUTPUT_DIR / "dataset_consolidado_hipe.csv", index=False, encoding="utf-8")
print(f"\n✅ Dataset consolidado guardado en: {OUTPUT_DIR / 'dataset_consolidado_hipe.csv'}")

print("\n===== VALIDACIÓN GENERAL =====")
print("Shape:", df.shape)
print("n_documentos:", df["document_id"].nunique())
print("Labels AT:", df["at"].value_counts(dropna=False).to_dict())
print("Labels isAt:", df["isAt"].value_counts(dropna=False).to_dict())

print("\n===== DISTRIBUCIONES GLOBALES =====")
dist_at = df["at"].value_counts().rename_axis("at").reset_index(name="count")
dist_isat = df["isAt"].value_counts().rename_axis("isAt").reset_index(name="count")
print(dist_at.to_string(index=False))
print(dist_isat.to_string(index=False))

print("\n" + "="*70)
print("Accuracy Profile:")
print("Ranking based on macro-averaged Recall (aka balanced accuracy) per relation type.\n")
print("Efficiency Profile:")
print("Ranking based on a composite metric balancing accuracy with:")
print("- Model size")
print("- Inference time")
print("- Hardware usage")
print("- Availability as open-source or low-cost system")
print("="*70)

# ============================================================
# 8) SPLIT
# ============================================================
train_docs, val_docs, doc_df = safe_group_stratified_split(df, test_size=0.30, random_state=SEED)
train_df = df[df["document_id"].isin(train_docs)].copy()
val_df = df[df["document_id"].isin(val_docs)].copy()

print_distributions("TRAIN (ANTES DE BALANCEO)", train_df)
print_distributions("VAL (NATURAL)", val_df)

dist_after = simulate_sampler_distribution(train_df, alpha=SAMPLER_ALPHA, draws=len(train_df), seed=SEED)
print("\n===== TRAIN (DESPUÉS DEL BALANCEO - simulación sampler por combo_label) =====")
print(dist_after.to_string(index=False))
dist_after.to_csv(OUTPUT_DIR / "train_dist_after_sampler_combo.csv", index=False, encoding="utf-8")

# ============================================================
# 9) INPUTS + TEMPORAL FEATS
# ============================================================
def build_packages_for_df(df_: pd.DataFrame, title: str) -> pd.DataFrame:
    packages = []
    for _, row in tqdm(df_.iterrows(), total=len(df_), desc=title):
        packages.append(build_paircentric_package(row))
    out = df_.copy().reset_index(drop=True)
    out["model_input"] = [p["model_input"] for p in packages]
    out["temporal_feats"] = [p["temporal_feats"] for p in packages]
    out["evidence_text"] = [p["evidence_text"] for p in packages]
    out["dateline_text"] = [p["dateline_text"] for p in packages]
    out["header_text"] = [p["header_text"] for p in packages]
    return out

print("\nConstruyendo model_input (train/val) + temporal_feats + NER features...")
train_df = build_packages_for_df(train_df, "Build train packages")
val_df = build_packages_for_df(val_df, "Build val packages")

train_df["input_char_len"] = train_df["model_input"].str.len()
val_df["input_char_len"] = val_df["model_input"].str.len()
print("\nLongitud model_input (chars):")
print("Train mean:", round(train_df["input_char_len"].mean(), 2), "| max:", int(train_df["input_char_len"].max()))
print("Val   mean:", round(val_df["input_char_len"].mean(), 2), "| max:", int(val_df["input_char_len"].max()))

# Labels
train_df["y_at"] = train_df["at"].map(AT_LABEL2ID).astype(int)
val_df["y_at"] = val_df["at"].map(AT_LABEL2ID).astype(int)
train_df["y_isat"] = train_df["isAt"].map(ISAT_LABEL2ID).astype(int)
val_df["y_isat"] = val_df["isAt"].map(ISAT_LABEL2ID).astype(int)

# ============================================================
# 10) TOKENIZER + Dataset
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def estimate_token_len_trunc(text: str) -> int:
    enc = tokenizer(text, truncation=True, max_length=512, add_special_tokens=True)
    return len(enc["input_ids"])

diag_sample = train_df["model_input"].sample(min(100, len(train_df)), random_state=SEED).tolist()
tok_lens = [estimate_token_len_trunc(x) for x in tqdm(diag_sample, desc="Diag tokens (trunc<=512)", leave=False)]
print(f"Tokens diag (n={len(tok_lens)}): mean={np.mean(tok_lens):.1f} | p95={np.percentile(tok_lens,95):.1f} | max={np.max(tok_lens)}")

class PairDataset(Dataset):
    def __init__(self, df_: pd.DataFrame, tokenizer_, max_len: int):
        self.df = df_.reset_index(drop=True).copy()
        self.tokenizer = tokenizer_
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        row = self.df.iloc[idx]
        enc = self.tokenizer(
            row["model_input"],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        temporal_feats = np.asarray(row["temporal_feats"], dtype=np.float32)
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "temporal_feats": torch.tensor(temporal_feats, dtype=torch.float32),
            "y_at": torch.tensor(int(row["y_at"]), dtype=torch.long),
            "y_isat": torch.tensor(int(row["y_isat"]), dtype=torch.float32),
        }

# ============================================================
# 11) MODELO CON T-PE TEMPORAL
# ============================================================
def masked_mean_pool(hidden_states: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).float()
    summed = (hidden_states * mask).sum(dim=1)
    denom = mask.sum(dim=1).clamp(min=1e-6)
    return summed / denom

class TemporalPositionalEncoding(nn.Module):
    """
    Componente geométrico del artículo:
    PE_i,2k = sin(i / 10000^(2k/d))
    PE_i,2k+1 = cos(i / 10000^(2k/d))
    """
    def __init__(self, d_model: int, max_len: int, n_temporal_features: int, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        position = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model)
        )
        pe = torch.zeros(max_len, d_model, dtype=torch.float32)
        pe[:, 0::2] = torch.sin(position * div_term)
        if d_model % 2 == 0:
            pe[:, 1::2] = torch.cos(position * div_term)
        else:
            pe[:, 1::2] = torch.cos(position * div_term[:pe[:, 1::2].shape[1]])
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)

        self.temporal_gate = nn.Sequential(
            nn.Linear(n_temporal_features, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
            nn.Sigmoid(),
        )
        self.temporal_context = nn.Sequential(
            nn.Linear(n_temporal_features, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
        )

    def forward(self, x: torch.Tensor, temporal_feats: Optional[torch.Tensor] = None) -> torch.Tensor:
        seq_len = x.size(1)
        pos = self.pe[:, :seq_len, :]

        if temporal_feats is not None:
            gate = self.temporal_gate(temporal_feats).unsqueeze(1)
            pos = pos * (1.0 + gate)
            ctx = self.temporal_context(temporal_feats).unsqueeze(1)
            x = x + (TEMPORAL_CONTEXT_SCALE * ctx)

        x = x + pos
        return self.dropout(x)

class TemporalSemanticSelfAttention(nn.Module):
    """
    Componente semántico del artículo:
    S(i,j) = exp(-||x_i - x_j||^2 / (2*sigma^2))
    Ese S(i,j) se usa como sesgo adicional en los attention scores.
    """
    def __init__(
        self,
        hidden_size: int,
        num_heads: int = 8,
        sim_dim: int = 64,
        dropout: float = 0.1,
        sigma_init: float = 1.0,
        semantic_weight_init: float = 0.35,
    ):
        super().__init__()
        assert hidden_size % num_heads == 0, "hidden_size debe ser divisible por num_heads"
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        self.scale = self.head_dim ** -0.5

        self.q_proj = nn.Linear(hidden_size, hidden_size)
        self.k_proj = nn.Linear(hidden_size, hidden_size)
        self.v_proj = nn.Linear(hidden_size, hidden_size)
        self.out_proj = nn.Linear(hidden_size, hidden_size)

        self.sim_proj = nn.Linear(hidden_size, sim_dim)
        self.dropout = nn.Dropout(dropout)

        self.log_sigma = nn.Parameter(torch.log(torch.tensor(float(sigma_init), dtype=torch.float32)))
        self.semantic_weight = nn.Parameter(torch.tensor(float(semantic_weight_init), dtype=torch.float32))

    def _shape(self, x: torch.Tensor) -> torch.Tensor:
        bsz, seq_len, _ = x.size()
        return x.view(bsz, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

    def _semantic_similarity(self, x: torch.Tensor, attention_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        z = self.sim_proj(x)  # [B,L,sim_dim]
        z2 = (z ** 2).sum(dim=-1, keepdim=True)
        dist2 = z2 + z2.transpose(1, 2) - 2.0 * torch.bmm(z, z.transpose(1, 2))
        dist2 = dist2.clamp(min=0.0)

        sigma = torch.exp(self.log_sigma).clamp(min=1e-3, max=10.0)
        sim = torch.exp(-dist2 / (2.0 * (sigma ** 2)))

        if attention_mask is not None:
            mask2d = attention_mask.unsqueeze(1) * attention_mask.unsqueeze(2)  # [B,L,L]
            sim = sim * mask2d.float()
        return sim

    def forward(self, x: torch.Tensor, attention_mask: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
        bsz, seq_len, _ = x.size()

        q = self._shape(self.q_proj(x))
        k = self._shape(self.k_proj(x))
        v = self._shape(self.v_proj(x))

        attn_scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale  # [B,H,L,L]

        semantic_bias = self._semantic_similarity(x, attention_mask=attention_mask).unsqueeze(1)  # [B,1,L,L]
        attn_scores = attn_scores + self.semantic_weight * semantic_bias

        if attention_mask is not None:
            key_mask = attention_mask[:, None, None, :].to(dtype=torch.bool)
            attn_scores = attn_scores.masked_fill(~key_mask, -1e4)

        attn_probs = F.softmax(attn_scores, dim=-1)
        attn_probs = self.dropout(attn_probs)

        context = torch.matmul(attn_probs, v)  # [B,H,L,D]
        context = context.transpose(1, 2).contiguous().view(bsz, seq_len, self.hidden_size)
        output = self.out_proj(context)
        return output, attn_probs

class TemporalTransformerRefinerLayer(nn.Module):
    def __init__(
        self,
        hidden_size: int,
        num_heads: int,
        sim_dim: int,
        dropout: float = 0.1,
        sigma_init: float = 1.0,
        semantic_weight_init: float = 0.35,
    ):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_size)
        self.attn = TemporalSemanticSelfAttention(
            hidden_size=hidden_size,
            num_heads=num_heads,
            sim_dim=sim_dim,
            dropout=dropout,
            sigma_init=sigma_init,
            semantic_weight_init=semantic_weight_init,
        )
        self.dropout1 = nn.Dropout(dropout)

        self.norm2 = nn.LayerNorm(hidden_size)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_size, hidden_size * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size * 4, hidden_size),
        )
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, attention_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        h = self.norm1(x)
        attn_out, _ = self.attn(h, attention_mask=attention_mask)
        x = x + self.dropout1(attn_out)

        h2 = self.norm2(x)
        ffn_out = self.ffn(h2)
        x = x + self.dropout2(ffn_out)
        return x

class TemporalTransformerRefiner(nn.Module):
    def __init__(
        self,
        hidden_size: int,
        num_layers: int,
        num_heads: int,
        sim_dim: int,
        dropout: float = 0.1,
        sigma_init: float = 1.0,
        semantic_weight_init: float = 0.35,
    ):
        super().__init__()
        self.layers = nn.ModuleList([
            TemporalTransformerRefinerLayer(
                hidden_size=hidden_size,
                num_heads=num_heads,
                sim_dim=sim_dim,
                dropout=dropout,
                sigma_init=sigma_init,
                semantic_weight_init=semantic_weight_init,
            )
            for _ in range(num_layers)
        ])
        self.final_norm = nn.LayerNorm(hidden_size)

    def forward(self, x: torch.Tensor, attention_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x, attention_mask=attention_mask)
        return self.final_norm(x)

class MultiTaskHipeTemporalModel(nn.Module):
    def __init__(self, model_name: str, n_temporal_features: int, dropout: float = 0.2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)

        # Bloque temporal T-PE adaptado
        self.temporal_pe = TemporalPositionalEncoding(
            d_model=hidden_size,
            max_len=TEMPORAL_MAX_POS,
            n_temporal_features=n_temporal_features,
            dropout=dropout,
        )

        self.temporal_refiner = TemporalTransformerRefiner(
            hidden_size=hidden_size,
            num_layers=TEMPORAL_REFINER_LAYERS,
            num_heads=TEMPORAL_REFINER_HEADS,
            sim_dim=TEMPORAL_SIM_DIM,
            dropout=dropout,
            sigma_init=TEMPORAL_SIGMA_INIT,
            semantic_weight_init=TEMPORAL_SEMANTIC_WEIGHT,
        )

        self.temporal_feat_proj = nn.Sequential(
            nn.Linear(n_temporal_features, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.fusion = nn.Sequential(
            nn.Linear(hidden_size * 3, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.head_at = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, 3)
        )
        self.head_isat = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, 1)
        )

    def forward(self, input_ids, attention_mask, temporal_feats):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        seq = out.last_hidden_state  # [B,L,H]

        # 1) Componente geométrico (sinusoidal)
        seq = self.temporal_pe(seq, temporal_feats=temporal_feats)

        # 2) Componente semántico + refinamiento temporal
        seq = self.temporal_refiner(seq, attention_mask=attention_mask)

        cls = seq[:, 0, :]
        mean_pool = masked_mean_pool(seq, attention_mask)
        temporal_ctx = self.temporal_feat_proj(temporal_feats)

        fused = self.fusion(torch.cat([cls, mean_pool, temporal_ctx], dim=-1))
        fused = self.dropout(fused)

        return self.head_at(fused), self.head_isat(fused).squeeze(-1)

# ============================================================
# 12) Evaluación
# ============================================================
@torch.no_grad()
def evaluate_model(model, loader, criterion_at, criterion_isat, device, at_thr_prob=0.34, at_thr_true=0.50, isat_thr=0.50):
    model.eval()
    total_loss, n_batches = 0.0, 0

    y_at_true, y_is_true = [], []
    logits_at_list, logits_is_list = [], []

    for batch in loader:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        temporal_feats = batch["temporal_feats"].to(device, non_blocking=True)
        tgt_at = batch["y_at"].to(device, non_blocking=True)
        tgt_is = batch["y_isat"].to(device, non_blocking=True)

        logits_at, logits_is = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            temporal_feats=temporal_feats
        )

        loss = criterion_at(logits_at, tgt_at) + criterion_isat(logits_is, tgt_is)
        total_loss += float(loss.item())
        n_batches += 1

        y_at_true.extend(tgt_at.detach().cpu().numpy().tolist())
        y_is_true.extend(tgt_is.detach().cpu().numpy().astype(int).tolist())

        logits_at_list.append(logits_at.detach().cpu())
        logits_is_list.append(logits_is.detach().cpu())

    logits_at_np = torch.cat(logits_at_list, dim=0).numpy()
    logits_is_np = torch.cat(logits_is_list, dim=0).numpy()

    probs_at = torch.softmax(torch.tensor(logits_at_np), dim=1).numpy()
    probs_is = torch.sigmoid(torch.tensor(logits_is_np)).numpy()

    pred_at = probs_to_at_preds_with_thresholds(probs_at, thr_prob=at_thr_prob, thr_true=at_thr_true)
    pred_is = (probs_is >= isat_thr).astype(int)

    y_at_np = np.array(y_at_true, dtype=int)
    y_is_np = np.array(y_is_true, dtype=int)

    ba_at = balanced_accuracy_score(y_at_np, pred_at)
    ba_is = balanced_accuracy_score(y_is_np, pred_is)

    return {
        "val_loss": total_loss / max(1, n_batches),
        "ba_at": float(ba_at),
        "ba_isat": float(ba_is),
        "score_macro": float((ba_at + ba_is) / 2.0),
        "y_at_true": y_at_np,
        "y_isat_true": y_is_np,
        "probs_at": probs_at,
        "probs_isat": probs_is,
        "pred_at": pred_at,
        "pred_isat": pred_is,
    }

# ============================================================
# 13) ENTRENAR 1 RUN
# ============================================================
def run_training(hparams: Dict[str, Any], epochs: int, run_tag: str) -> Dict[str, Any]:
    global MAX_LEN, LR_ENCODER, LR_HEADS, DROPOUT, WEIGHT_DECAY, SAMPLER_ALPHA, ISAT_POS_WEIGHT

    MAX_LEN = int(hparams["max_len"])
    LR_ENCODER = float(hparams["lr_encoder"])
    LR_HEADS = float(hparams["lr_heads"])
    DROPOUT = float(hparams["dropout"])
    WEIGHT_DECAY = float(hparams["weight_decay"])
    SAMPLER_ALPHA = float(hparams["sampler_alpha"])
    ISAT_POS_WEIGHT = float(hparams["isat_pos_weight"])

    train_ds = PairDataset(train_df, tokenizer, max_len=MAX_LEN)
    val_ds = PairDataset(val_df, tokenizer, max_len=MAX_LEN)

    train_sampler = soft_balance_sampler_from_combo(train_df, alpha=SAMPLER_ALPHA)

    dist_after = simulate_sampler_distribution(train_df, alpha=SAMPLER_ALPHA, draws=len(train_df), seed=SEED)
    dist_after.to_csv(OUTPUT_DIR / f"dist_after_sampler_{run_tag}.csv", index=False, encoding="utf-8")

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, sampler=train_sampler,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
    )

    model = MultiTaskHipeTemporalModel(
        MODEL_NAME,
        n_temporal_features=len(TEMPORAL_FEATURE_NAMES),
        dropout=DROPOUT
    ).to(device)

    w_at = torch.tensor(
        [AT_CLASS_WEIGHTS_SOFT["FALSE"], AT_CLASS_WEIGHTS_SOFT["PROBABLE"], AT_CLASS_WEIGHTS_SOFT["TRUE"]],
        dtype=torch.float32, device=device
    )
    criterion_at = nn.CrossEntropyLoss(weight=w_at)
    criterion_isat = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(ISAT_POS_WEIGHT, dtype=torch.float32, device=device)
    )

    encoder_params = list(model.encoder.named_parameters())
    temporal_block_params = [
        (n, p) for n, p in model.named_parameters()
        if not n.startswith("encoder.")
    ]

    optimizer = torch.optim.AdamW(
        [
            {"params": [p for _, p in encoder_params], "lr": LR_ENCODER, "weight_decay": WEIGHT_DECAY},
            {"params": [p for _, p in temporal_block_params], "lr": LR_HEADS, "weight_decay": WEIGHT_DECAY},
        ]
    )

    num_training_steps = epochs * len(train_loader)
    num_warmup_steps = int(0.10 * num_training_steps)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps)

    scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED) if torch.cuda.is_available() else None

    best_score = -1.0
    best_state = None
    patience_count = 0
    history = []

    cur_at_thr_prob = AT_THR_PROB_INIT
    cur_at_thr_true = AT_THR_TRUE_INIT
    cur_isat_thr = ISAT_THR_INIT

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        pbar = tqdm(train_loader, total=len(train_loader), desc=f"{run_tag} Epoch {epoch+1}/{epochs}", leave=False)

        for batch in pbar:
            optimizer.zero_grad(set_to_none=True)
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            temporal_feats = batch["temporal_feats"].to(device, non_blocking=True)
            y_at = batch["y_at"].to(device, non_blocking=True)
            y_isat = batch["y_isat"].to(device, non_blocking=True)

            if torch.cuda.is_available():
                with torch.amp.autocast("cuda", enabled=AMP_ENABLED):
                    logits_at, logits_isat = model(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        temporal_feats=temporal_feats
                    )
                    loss = criterion_at(logits_at, y_at) + criterion_isat(logits_isat, y_isat)

                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
            else:
                logits_at, logits_isat = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    temporal_feats=temporal_feats
                )
                loss = criterion_at(logits_at, y_at) + criterion_isat(logits_isat, y_isat)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                optimizer.step()

            scheduler.step()
            running_loss += float(loss.item())
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        train_loss = running_loss / max(1, len(train_loader))
        eval_out = evaluate_model(
            model, val_loader, criterion_at, criterion_isat, device,
            at_thr_prob=cur_at_thr_prob, at_thr_true=cur_at_thr_true, isat_thr=cur_isat_thr
        )

        history.append({
            "epoch": epoch + 1,
            "train_loss": float(train_loss),
            "val_loss": float(eval_out["val_loss"]),
            "ba_at": float(eval_out["ba_at"]),
            "ba_isat": float(eval_out["ba_isat"]),
            "score_macro": float(eval_out["score_macro"]),
        })

        if eval_out["score_macro"] > best_score:
            best_score = float(eval_out["score_macro"])
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                break

    history_df = pd.DataFrame(history)

    if best_state is not None:
        model.load_state_dict(best_state)

    val_raw = evaluate_model(
        model, val_loader, criterion_at, criterion_isat, device,
        at_thr_prob=AT_THR_PROB_INIT, at_thr_true=AT_THR_TRUE_INIT, isat_thr=ISAT_THR_INIT
    )

    best_at_thr = tune_at_thresholds_for_ba(val_raw["y_at_true"], val_raw["probs_at"])
    best_isat_thr = tune_isat_threshold_for_ba(val_raw["y_isat_true"], val_raw["probs_isat"])

    final_eval = evaluate_model(
        model, val_loader, criterion_at, criterion_isat, device,
        at_thr_prob=best_at_thr["thr_prob"], at_thr_true=best_at_thr["thr_true"], isat_thr=best_isat_thr["thr"]
    )

    return {
        "model": model,
        "history_df": history_df,
        "final_eval": final_eval,
        "best_at_thr": best_at_thr,
        "best_isat_thr": best_isat_thr,
        "val_loader": val_loader
    }

# ============================================================
# 14) OPTUNA
# ============================================================
def objective(trial: optuna.Trial) -> float:
    hparams = {
        "max_len": trial.suggest_int("max_len", 224, 384, step=32),
        "lr_encoder": trial.suggest_float("lr_encoder", 5e-6, 2e-5, log=True),
        "lr_heads": trial.suggest_float("lr_heads", 1e-5, 8e-5, log=True),
        "dropout": trial.suggest_float("dropout", 0.20, 0.40),
        "weight_decay": trial.suggest_float("weight_decay", 0.005, 0.05),
        "sampler_alpha": trial.suggest_float("sampler_alpha", 0.30, 0.80),
        "isat_pos_weight": trial.suggest_float("isat_pos_weight", 2.3, 3.8),
    }
    out = run_training(hparams, epochs=EPOCHS_OPTUNA, run_tag=f"trial_{trial.number:03d}")
    score = out["final_eval"]["score_macro"]
    trial.report(score, step=1)
    return score

best_hparams = {
    "max_len": MAX_LEN,
    "lr_encoder": LR_ENCODER,
    "lr_heads": LR_HEADS,
    "dropout": DROPOUT,
    "weight_decay": WEIGHT_DECAY,
    "sampler_alpha": SAMPLER_ALPHA,
    "isat_pos_weight": ISAT_POS_WEIGHT,
}

if USE_OPTUNA:
    study = optuna.create_study(
        direction="maximize",
        sampler=TPESampler(seed=SEED),
        pruner=MedianPruner(n_startup_trials=5)
    )
    study.optimize(objective, n_trials=N_TRIALS, timeout=OPTUNA_TIMEOUT, show_progress_bar=True)
    best_hparams = study.best_trial.params
    with open(OUTPUT_DIR / "best_optuna_params.json", "w", encoding="utf-8") as f:
        json.dump(best_hparams, f, ensure_ascii=False, indent=2)

    print("\n✅ OPTUNA mejor score:", study.best_value)
    print("✅ Mejores hiperparámetros:", json.dumps(best_hparams, indent=2, ensure_ascii=False))

# ============================================================
# 15) ENTRENAMIENTO FINAL
# ============================================================
print("\n" + "="*80)
print("ENTRENAMIENTO FINAL con mejores hiperparámetros")
print("="*80)

start_final = time.perf_counter()
final_out = run_training(best_hparams, epochs=EPOCHS, run_tag="FINAL")
train_elapsed = time.perf_counter() - start_final

model = final_out["model"]
history_df = final_out["history_df"]
final_eval = final_out["final_eval"]
best_at_thr = final_out["best_at_thr"]
best_isat_thr = final_out["best_isat_thr"]
val_loader = final_out["val_loader"]

history_df.to_csv(OUTPUT_DIR / "training_history.csv", index=False, encoding="utf-8")

# ============================================================
# 16) METRICAS promedio + perfiles
# ============================================================
avg_ba_at_epochs = float(history_df["ba_at"].mean())
avg_ba_isat_epochs = float(history_df["ba_isat"].mean())
avg_ba_macro_epochs = float((avg_ba_at_epochs + avg_ba_isat_epochs) / 2.0)
avg_train_loss_epochs = float(history_df["train_loss"].mean())
avg_val_loss_epochs = float(history_df["val_loss"].mean())

df_accuracy_profile = pd.DataFrame([{
    "BA_at_avg_epochs": avg_ba_at_epochs,
    "BA_isAt_avg_epochs": avg_ba_isat_epochs,
    "BA_macro_avg_epochs": avg_ba_macro_epochs,
    "train_loss_avg_epochs": avg_train_loss_epochs,
    "val_loss_avg_epochs": avg_val_loss_epochs,
    "BA_at_final_tuned": float(final_eval["ba_at"]),
    "BA_isAt_final_tuned": float(final_eval["ba_isat"]),
    "BA_macro_final_tuned": float(final_eval["score_macro"]),
    "thr_at_prob": float(best_at_thr["thr_prob"]),
    "thr_at_true": float(best_at_thr["thr_true"]),
    "thr_isAt": float(best_isat_thr["thr"]),
}])

model_size_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / (1024 ** 2)

model.eval()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

start_inf = time.perf_counter()
n_samples = 0
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        temporal_feats = batch["temporal_feats"].to(device, non_blocking=True)
        _ = model(input_ids=input_ids, attention_mask=attention_mask, temporal_feats=temporal_feats)
        n_samples += input_ids.size(0)

if torch.cuda.is_available():
    torch.cuda.synchronize()

elapsed_inf = time.perf_counter() - start_inf
time_per_sample_ms = (elapsed_inf / max(1, n_samples)) * 1000.0

hardware_info = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
using_cuda = torch.cuda.is_available()
opensource_flag = 1

eff_proxy = compute_efficiency_proxy(
    acc_macro=avg_ba_macro_epochs,
    model_size_mb=float(model_size_mb),
    time_per_sample_ms=float(time_per_sample_ms),
    using_cuda=using_cuda,
    opensource_flag=opensource_flag
)

df_efficiency_profile = pd.DataFrame([{
    "model_size_mb": float(model_size_mb),
    "time_per_sample_ms": float(time_per_sample_ms),
    "hardware": hardware_info,
    "open_source_low_cost": int(opensource_flag),
    "accuracy_macro_avg_epochs": float(avg_ba_macro_epochs),
    "efficiency_proxy": float(eff_proxy),
}])

# ============================================================
# 17) REPORTES + MATRICES
# ============================================================
print("\n" + "="*80)
print("RESULTADO FINAL (MEJOR MODELO + THRESHOLDS TUNEADOS)")
print("="*80)
print(f"Balanced Accuracy final (AT)      : {final_eval['ba_at']:.4f}")
print(f"Balanced Accuracy final (isAt)    : {final_eval['ba_isat']:.4f}")
print(f"Macro final (AT/isAt)             : {final_eval['score_macro']:.4f}")
print(f"Thresholds -> AT(prob={best_at_thr['thr_prob']:.2f}, true={best_at_thr['thr_true']:.2f}) | isAt(thr={best_isat_thr['thr']:.2f})")

print("\nReporte de clasificación - AT")
print(classification_report(final_eval["y_at_true"], final_eval["pred_at"], labels=[0,1,2], target_names=AT_LABELS, digits=4, zero_division=0))

print("\nReporte de clasificación - isAt")
print(classification_report(final_eval["y_isat_true"], final_eval["pred_isat"], labels=[0,1], target_names=ISAT_LABELS, digits=4, zero_division=0))

cm_at = confusion_matrix(final_eval["y_at_true"], final_eval["pred_at"], labels=[0,1,2])
cm_isat = confusion_matrix(final_eval["y_isat_true"], final_eval["pred_isat"], labels=[0,1])
pd.DataFrame(cm_at, index=AT_LABELS, columns=AT_LABELS).to_csv(OUTPUT_DIR / "cm_at.csv", encoding="utf-8")
pd.DataFrame(cm_isat, index=ISAT_LABELS, columns=ISAT_LABELS).to_csv(OUTPUT_DIR / "cm_isAt.csv", encoding="utf-8")

# ============================================================
# 18) CURVA DE PÉRDIDA
# ============================================================
if HAS_MPL:
    plt.figure(figsize=(8, 5))
    plt.plot(history_df["epoch"], history_df["train_loss"], marker="o", label="Train Loss")
    plt.plot(history_df["epoch"], history_df["val_loss"], marker="o", label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Curva de pérdida (Train vs Val)")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "loss_curve_train_val.png", dpi=150)
    plt.show()

# ============================================================
# 19) IMPRIMIR LAS 2 TABLAS
# ============================================================
print("\n" + "="*80)
print("TABLA 1 — Accuracy Profile (Balanced Accuracy / Macro Recall)")
print("="*80)
print(df_accuracy_profile.to_string(index=False))

print("\n" + "="*80)
print("TABLA 2 — Efficiency Profile (composite proxy)")
print("="*80)
print(df_efficiency_profile.to_string(index=False))

df_accuracy_profile.to_csv(OUTPUT_DIR / "accuracy_profile_table.csv", index=False, encoding="utf-8")
df_efficiency_profile.to_csv(OUTPUT_DIR / "efficiency_profile_table.csv", index=False, encoding="utf-8")

# ============================================================
# 20) RESUMEN FINAL
# ============================================================
summary = {
    "best_optuna_params": best_hparams,
    "training_minutes": float(train_elapsed / 60.0),
    "accuracy_profile": df_accuracy_profile.to_dict(orient="records")[0],
    "efficiency_profile": df_efficiency_profile.to_dict(orient="records")[0],
    "global_at_distribution": df["at"].value_counts().to_dict(),
    "global_isAt_distribution": df["isAt"].value_counts().to_dict(),
    "temporal_feature_names": TEMPORAL_FEATURE_NAMES,
    "temporal_tpe": {
        "enabled": USE_TEMPORAL_TPE,
        "layers": TEMPORAL_REFINER_LAYERS,
        "heads": TEMPORAL_REFINER_HEADS,
        "sim_dim": TEMPORAL_SIM_DIM,
        "sigma_init": TEMPORAL_SIGMA_INIT,
        "semantic_weight_init": TEMPORAL_SEMANTIC_WEIGHT,
    }
}
with open(OUTPUT_DIR / "run_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("\n✅ Proceso completado.")
print("Archivos generados en:", OUTPUT_DIR.resolve())

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
